# 01 · NHANES dataset

Builds the training data the per-panel models will use, from the **NHANES 2017–March 2020
pre-pandemic** cycle (CDC, public domain):

1. Download the lab, demographic, body-measure and liver-elastography files (cached in `ml/data/nhanes/`).
2. Check every `nhanes` variable named in `backend/biomarkers.py` against the files and their
   labelled units — this is the codebook check the registry asks for.
3. Convert values to the registry's canonical units, so a model trained here sees inputs exactly
   like `analysis.py` gives it from a parsed report.
4. Add the values the app derives when a report leaves them out (Non-HDL, Globulin, ratios).
5. Save one table of adults with an exam, plus how much of each panel's inputs NHANES can supply.

Nothing here is committed with row-level data: `ml/data/` is git-ignored.

In [1]:
import sys
from pathlib import Path

ML_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ML_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nhanes  # also puts backend/ on the path
from biomarkers import PANELS

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
DATA = ML_DIR / "data"


## Load and join the cycle

In [2]:
raw, labels = nhanes.load_cycle()
print(f"{len(raw):,} participants, {raw.shape[1]} columns from {len(nhanes.PRE_PANDEMIC['files'])} files")

15,560 participants, 198 columns from 18 files


## Registry ↔ NHANES check

`ok` means the variable exists and its labelled unit converts to the registry's canonical unit.
`not in these files` means NHANES measured it in another cycle (e.g. thyroid in 2007–2012), not
that the name is wrong.

In [3]:
mapping = nhanes.registry_mapping(labels)
print(mapping["status"].value_counts().to_string())
mapping[["biomarker", "nhanes", "file", "nhanes_unit", "registry_unit", "factor", "status"]]

status
ok                    59
not in these files    12


,biomarker,nhanes,file,nhanes_unit,registry_unit,factor,status
0,White Blood Cells,LBXWBCSI,P_CBC,10^3/uL,10^3/uL,1.00,ok
1,Red Blood Cells,LBXRBCSI,P_CBC,10^6/uL,10^6/uL,1.00,ok
2,Hemoglobin,LBXHGB,P_CBC,g/dL,g/dL,1.00,ok
3,Hematocrit,LBXHCT,P_CBC,%,%,1.00,ok
4,MCV,LBXMCVSI,P_CBC,fL,fL,1.00,ok
5,MCH,LBXMCHSI,P_CBC,pg,pg,1.00,ok
6,MCHC,LBXMC,P_CBC,g/dL,g/dL,1.00,ok
7,RDW,LBXRDW,P_CBC,%,%,1.00,ok
8,Platelets,LBXPLTSI,P_CBC,10^3/uL,10^3/uL,1.00,ok
9,MPV,LBXMPSI,P_CBC,fL,fL,1.00,ok


## Adults with an exam, in registry units

In [4]:
# Registry units, plus the values the app derives (Non-HDL, Globulin, ...) the same way.
values = nhanes.add_derived(nhanes.to_registry(raw, mapping))
demo = nhanes.demographics(raw)
liver_scan = pd.DataFrame({
    # Elastography: complete exam (status 1); CAP = liver fat, stiffness = scarring.
    "elastography_complete": raw["LUAXSTAT"] == 1,
    "cap_dbm": raw["LUXCAPM"],
    "stiffness_kpa": raw["LUXSMED"],
    "stiffness_iqr_ratio": raw["LUXSIQRM"],  # IQR / median, in percent
    "bmi": raw["BMXBMI"],
})
adults = pd.concat([raw[["SEQN"]], demo, values, liver_scan], axis=1)
adults = adults[(adults["age"] >= 18) & adults["weight"].gt(0)].reset_index(drop=True)
DATA.mkdir(exist_ok=True)
adults.to_parquet(DATA / "nhanes_2017_2020_adults.parquet")
print(f"{len(adults):,} adults saved to ml/data/nhanes_2017_2020_adults.parquet")
adults[["age"]].describe().T.join(adults["sex"].value_counts().rename_axis(None).to_frame().T.reset_index(drop=True).set_axis(["age"]))

8,965 adults saved to ml/data/nhanes_2017_2020_adults.parquet


,count,mean,std,min,25%,50%,75%,max,female,male
age,8965.0,49.465142,18.457564,18.0,33.0,50.0,64.0,80.0,4608,4357


## What each panel's model could learn from

For each panel: how many of its `inputs` NHANES supplies, and how many adults have enough of
the panel's required tests for `analysis.py` to analyze it.

In [5]:
from analysis import _is_available

rows = []
available = set(values.columns)
for panel_id, panel in PANELS.items():
    mapped = [n for n in panel["inputs"] if n in available]
    if not mapped:
        rows.append({"panel": panel_id, "inputs": len(panel["inputs"]), "in NHANES": 0, "adults analyzable": 0})
        continue
    present = adults[mapped].notna()
    analyzable = present.apply(lambda row: _is_available(panel_id, set(row.index[row])), axis=1).sum()
    rows.append({"panel": panel_id, "inputs": len(panel["inputs"]), "in NHANES": len(mapped), "adults analyzable": int(analyzable),
                 "missing from NHANES": ", ".join(n for n in panel["inputs"] if n not in available)})
coverage = pd.DataFrame(rows).set_index("panel")
coverage

,inputs,in NHANES,adults analyzable,missing from NHANES
panel,,,,
cbc,12,11,8510,Reticulocytes
differential,10,10,8506,
cmp,19,18,8257,eGFR
bmp,11,10,8258,eGFR
lipid,8,5,8298,"VLDL Cholesterol, Cholesterol/HDL Ratio, LDL/H..."
liver,11,9,8255,"Bilirubin, Direct, Bilirubin, Indirect"
kidney,9,6,8255,"Urea, eGFR, Cystatin C"
thyroid,6,0,0,NaN
diabetes,4,3,8472,Estimated Average Glucose


Panels with thousands of analyzable adults (CBC, differential, CMP/BMP, lipid, liver, kidney,
diabetes, iron, inflammation via hs-CRP) can be modelled from this cycle alone. Thyroid, vitamins
and most hormones need earlier cycles (e.g. 2011–2012), which use the same variable names with a
different file suffix (`_G`).